<a href="https://www.kaggle.com/code/zeeshang25ait2135/mlops-group14-experiment-2?scriptVersionId=325252414" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# MLOps Group Project - Experiment 1: Baseline Fine-Tuning
**Task 3 & Task 4: Model Selection, Kaggle Training, and W&B Tracking**

## 1. Environment Setup
To build our end-to-end pipeline, we need several libraries: `transformers` and `datasets` for handling the Hugging Face model and data, `evaluate` to calculate our accuracy metrics, and `wandb` to track our experiment parameters and metrics securely.

In [ ]:
!pip install -q transformers datasets evaluate wandb scikit-learn

## 2. Secure Authentication
**Justification:** Hardcoding API keys is a bad MLOps practice. We are using `Kaggle Secrets` to securely fetch our Weights & Biases (W&B) and Hugging Face tokens. W&B will act as our central dashboard for tracking loss and accuracy, while the Hugging Face login prepares our environment for Task 5 (Pushing the model to the Hub).

In [ ]:
import os
import wandb
import warnings
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, whoami

warnings.filterwarnings("ignore", category=SyntaxWarning)

print("Fetching secrets from Kaggle...")
user_secrets = UserSecretsClient()
wandb_key = user_secrets.get_secret("WANDB_API_KEY") 
hf_token = user_secrets.get_secret("HF_TOKEN")       

# Weights & Biases Authentication & Validation
print("\nConnecting to Weights & Biases...")
os.environ["WANDB_API_KEY"] = wandb_key
wandb_success = wandb.login()

if wandb_success:
    print("✅ W&B Connection Established Successfully!")
else:
    print("❌ W&B Connection Failed. Please check your WANDB_API_KEY.")

os.environ["WANDB_PROJECT"] = "mlops-emotion-classification" 
os.environ["WANDB_LOG_MODEL"] = "checkpoint" 

# Hugging Face Authentication & Validation
print("\nConnecting to Hugging Face...")
login(token=hf_token)

try:
    # whoami() actively pings Hugging Face to verify the token works
    hf_user_info = whoami()
    print(f"✅ Hugging Face Connection Established Successfully! Logged in as: {hf_user_info['name']}")
except Exception as e:
    print(f"❌ Hugging Face Connection Failed. Please check your HF_TOKEN. Error: {e}")

## 3. Data Loading & Model Selection (Task 3)

**Justification:** 
* **Model Selection:** We selected `distilbert-base-uncased`. As per the Hugging Face model card, DistilBERT is a smaller, faster, and lighter version of BERT. At roughly 260MB (uncompressed), it is highly compact, meaning it trains quickly and fits easily within Kaggle's free T4 GPU memory limits, fulfilling the assignment's "compact model" constraint.
* **Data Preparation:** We are using the `dair-ai/emotion` dataset. To iterate quickly and avoid exhausting Kaggle quotas, we are taking a random, seeded subset of 5,000 training samples and 1,000 validation samples. We are also explicitly setting `num_labels=6` to match our `id2label.json` mapping.

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification

print("Loading dataset...")
# Load the Emotion Dataset
dataset = load_dataset("dair-ai/emotion")

# Select compact model
model_name = "distilbert-base-uncased"
print(f"Loading Tokenizer and Model: {model_name}...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
# We set num_labels=6 because our id2label.json has 6 emotion classes
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=6)

# Tokenization function
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

print("Tokenizing data...")
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Downsampling for faster experimentation
print("Downsampling for Experiment 1...")
small_train = tokenized_datasets["train"].shuffle(seed=42).select(range(5000))
small_eval = tokenized_datasets["validation"].shuffle(seed=42).select(range(1000))

print(f"✅ Data Preparation Complete! Training samples: {len(small_train)} | Validation samples: {len(small_eval)}")

## 4. Experiment 2: Hyperparameter Tuning (Task 4)

**Justification:** To fulfill the multi-version experiment requirement, we are modifying our baseline hyperparameters to see if we can achieve faster convergence or better generalization:
* **Learning Rate:** Increased from `2e-5` to `5e-5`. A higher learning rate might help the model find the optimal weights faster.
* **Epochs:** Increased from `3` to `4` to give the model slightly more time to learn from the higher learning rate.
* **Weight Decay:** Increased from `0.01` to `0.05` to introduce stronger regularization and prevent the model from overfitting on our small 5k sample dataset.

We update the W&B `run_name` to `distilbert-exp2-tuned` so it appears as a distinct comparison run on our dashboard.

In [ ]:
import evaluate
import numpy as np
from transformers import TrainingArguments, Trainer

print("Loading evaluation metrics...")
# Define Accuracy Metric
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

print("Configuring Training Arguments for Experiment 2...")
# Tuned Hyperparameters (Experiment 2)
training_args = TrainingArguments(
    output_dir="./results_exp2",
    learning_rate=5e-5,               # CHANGED: Higher learning rate
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,               # CHANGED: Extra epoch
    weight_decay=0.05,                # CHANGED: Stronger regularization
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    report_to="wandb",                
    run_name="distilbert-exp2-tuned", # CHANGED: New W&B run name
    push_to_hub=False                 
)

print("Initializing Trainer...")
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_eval,
    compute_metrics=compute_metrics,
)

print("🚀 Starting Experiment 2 Training...")
# Execute Training
trainer.train()

print("Training complete! Syncing final logs to Weights & Biases...")
# Close the W&B run to finalize logs
import wandb
wandb.finish()

print("✅ Experiment 2 Finished! Click the wandb link above to view your dashboard.")

## 5. Model Deployment to Hugging Face Hub (Task 5)

**Justification:** After analyzing our W&B dashboard, Experiment 2 achieved a superior validation accuracy (93.7%) compared to the baseline (90.4%). Therefore, we select the Experiment 2 model as our final production candidate. We are pushing both the model weights and the tokenizer to the Hugging Face Hub so it can be easily downloaded and containerized in our Docker deployment (Task 6).

In [ ]:
# Name the repository
hf_repo_name = "mlops-emotion-distilbert-group14"

print(f"🚀 Pushing the winning model to Hugging Face Hub: {hf_repo_name}...")

# Push the trained model weights
model.push_to_hub(hf_repo_name)

# Push the tokenizer so it can process text during inference
tokenizer.push_to_hub(hf_repo_name)

print("✅ Model Deployment Complete! You can now view it on your Hugging Face profile.")